# 01 — Build paper-brief evaluation corpus

Freeze usable `full_text_plain` from **archived local Papers** into `data/paper_brief_evaluation/corpus/`.

**Prerequisite:** the app Postgres (`just up`) already has `Paper` rows. This notebook queries those rows. It does not archive and it does not fetch full text. Run it with `just notebooks`, not `just sandbox`.

**Eligible paper:** `usable_full_text_plain(paper.full_text_plain)` is set. `PaperBrief` is ignored. If `corpus/{DOI_FILE}.txt` already exists, skip that paper (do not overwrite) and say so.

Domain calls: load `Paper` rows, then `usable_full_text_plain`. Do not import `paper_reviewer.flows`. Do not call `inform_source_record` or `inform_full_text`. This notebook does not write `PaperBrief`.

**`manifest.jsonl`:** sidecar index for this corpus. Each `.txt` is `full_text_plain` only (no YAML header). Notebook 02 does not query Postgres; it reads this file for `doi`, `title`, `journal`, `published_year`, and `filename`, then loads `corpus/{filename}` into `generate_paper_brief_content`. Notebook 03 does **not** read the manifest; it rebuilds the `.txt` name from the DOI. After this run, rewrite the manifest from every `.txt` still in `corpus/`, with title / journal / year from the database. A `.txt` with no matching `Paper` is omitted and reported.

**Git:** corpus files (`.txt` + `manifest.jsonl`) and later `{run_id}/` results under `data/paper_brief_evaluation/` are tracked so you can commit them. They stay out of the production image (`.dockerignore`).

Input is every `Paper` row in local Postgres. There is no DOI list.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

from sqlalchemy import select

from paper_reviewer.db import create_db_engine, create_session_factory, session_scope
from paper_reviewer.ingest.pubmed.pmc_cloud import usable_full_text_plain
from paper_reviewer.models.paper import Paper


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "src" / "paper_reviewer").is_dir() and (
            candidate / "pyproject.toml"
        ).is_file():
            return candidate
    raise RuntimeError(
        "Cannot find the repo root. Start Jupyter with `just notebooks` "
        "so the kernel can see /workspace."
    )


REPO_ROOT = repo_root()
CORPUS_DIR = REPO_ROOT / "data" / "paper_brief_evaluation" / "corpus"
MANIFEST_PATH = CORPUS_DIR / "manifest.jsonl"
SESSION_FACTORY = create_session_factory(create_db_engine())
print(f"repo root: {REPO_ROOT}")
print(f"corpus dir: {CORPUS_DIR}")

repo root: /workspace
corpus dir: /workspace/data/paper_brief_evaluation/corpus


In [2]:
def normalize_doi(raw: str) -> str:
    return raw.strip().upper()


def corpus_filename(doi: str) -> str:
    return f"{doi.replace('/', '_')}.txt"


def first_filename_collision(dois: list[str]) -> tuple[str, str] | None:
    by_name: dict[str, str] = {}
    for doi in dois:
        name = corpus_filename(doi)
        owner = by_name.get(name)
        if owner is not None and owner != doi:
            return owner, doi
        by_name[name] = doi
    return None

In [3]:
papers: list[dict] = []
with session_scope(SESSION_FACTORY) as session:
    for paper in session.scalars(select(Paper).order_by(Paper.doi)):
        doi = normalize_doi(paper.doi)
        papers.append(
            {
                "doi": doi,
                "title": paper.title,
                "journal": paper.journal,
                "published_year": paper.published_year,
                "filename": corpus_filename(doi),
                "body": usable_full_text_plain(paper.full_text_plain),
            }
        )

collision = first_filename_collision([row["doi"] for row in papers])
if collision is not None:
    left, right = collision
    raise RuntimeError(
        "Corpus filenames collide for "
        f"{left!r} and {right!r} -> {corpus_filename(left)}. "
        "Step 1 stopped. Previous corpus files were not changed."
    )

papers_by_filename = {row["filename"]: row for row in papers}

accepted: list[dict] = []
skipped_already: list[str] = []
skipped_no_text: list[str] = []
errors: list[tuple[str, str]] = []

for row in papers:
    doi = row["doi"]
    filename = row["filename"]
    try:
        if row["body"] is None:
            print(f"SKIP {doi}: no usable full text")
            skipped_no_text.append(doi)
            continue
        dest = CORPUS_DIR / filename
        if dest.is_file():
            print(f"SKIP {doi}: already in corpus -> {filename}")
            skipped_already.append(doi)
            continue
        accepted.append(row)
        print(f"ACCEPT {doi} -> {filename}")
    except Exception as exc:
        print(f"ERROR {doi}: {exc}")
        errors.append((doi, str(exc)))

if accepted:
    CORPUS_DIR.mkdir(parents=True, exist_ok=True)
    for row in accepted:
        (CORPUS_DIR / row["filename"]).write_text(row["body"], encoding="utf-8")
    print(f"Wrote {len(accepted)} new .txt file(s)")
else:
    print("No new papers accepted. Existing corpus files were not overwritten.")

txt_files = (
    sorted(path for path in CORPUS_DIR.glob("*.txt") if path.is_file())
    if CORPUS_DIR.is_dir()
    else []
)
if txt_files:
    CORPUS_DIR.mkdir(parents=True, exist_ok=True)
    manifest_rows: list[dict] = []
    skipped_orphan: list[str] = []
    for path in txt_files:
        meta = papers_by_filename.get(path.name)
        if meta is None:
            print(f"SKIP manifest {path.name}: no Paper row for this file")
            skipped_orphan.append(path.name)
            continue
        manifest_rows.append(
            {
                "doi": meta["doi"],
                "title": meta["title"],
                "journal": meta["journal"],
                "published_year": meta["published_year"],
                "filename": path.name,
            }
        )
    manifest_rows.sort(key=lambda item: item["doi"])
    with MANIFEST_PATH.open("w", encoding="utf-8") as handle:
        for record in manifest_rows:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"Wrote {len(manifest_rows)} manifest line(s) to {MANIFEST_PATH}")
else:
    skipped_orphan = []
    print("No .txt files in corpus. Manifest was not written.")

print("---")
print(f"papers in database: {len(papers)}")
print(f"accepted (new .txt): {len(accepted)}")
print(f"skipped (already in corpus): {len(skipped_already)}")
print(f"skipped (no usable text): {len(skipped_no_text)}")
print(f"skipped (corpus file with no Paper): {len(skipped_orphan)}")
print(f"errors: {len(errors)}")

ACCEPT 10.1002/ECE3.74085 -> 10.1002_ECE3.74085.txt
SKIP 10.1002/JMV.71049: no usable full text
SKIP 10.1002/MDC3.70417: no usable full text
ACCEPT 10.1002/PROT.70113 -> 10.1002_PROT.70113.txt
ACCEPT 10.1002/RMV.70176 -> 10.1002_RMV.70176.txt
SKIP 10.1002/RMV.70182: no usable full text
ACCEPT 10.1002/VMS3.71125 -> 10.1002_VMS3.71125.txt
SKIP 10.1007/S00044-026-03539-0: no usable full text
SKIP 10.1007/S00284-026-04938-7: no usable full text
SKIP 10.1007/S00347-026-02470-4: no usable full text
SKIP 10.1007/S11250-026-05070-1: no usable full text
SKIP 10.1007/S11686-026-01307-Z: no usable full text
SKIP 10.1007/S11686-026-01320-2: no usable full text
SKIP 10.1007/S11739-026-04392-0: no usable full text
SKIP 10.1007/S13365-026-01310-0: no usable full text
SKIP 10.1007/S13365-026-01317-7: no usable full text
SKIP 10.1007/S15010-026-02919-3: no usable full text
SKIP 10.1016/J.ACTATROPICA.2026.108228: no usable full text
SKIP 10.1016/J.AJO.2026.03.015: no usable full text
SKIP 10.1016/J.AJT.

After a successful build, commit the corpus (and later `{run_id}/` results) if you want them in the repository:

```bash
git add data/paper_brief_evaluation/
```

Production images still exclude `data/` via `.dockerignore`.